# Kaggle Music Generator — Persistent ACE-Step Session
One Kaggle run watches public GitHub Issues and accepts multiple music requests. ACE-Step HTTP server uses --keep-loaded. Kaggle performs anonymous GET requests only; GitHub Actions handles all authenticated writes.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
print('Python:', sys.version, flush=True)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
subprocess.run(['apt-get','update','-qq'], check=True)
subprocess.run(['apt-get','install','-y','-qq','build-essential','git','curl','ca-certificates','tar','coreutils','libgomp1'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','hf','six'], check=True)
subprocess.run(['hf','--help'], check=True, stdout=subprocess.DEVNULL)

In [ ]:
engine_dir=Path('/kaggle/working/acestep.cpp')
if not engine_dir.exists(): subprocess.run(['git','clone','--depth','1','--branch','v0.1.2','https://github.com/pwilkin/acestep.cpp.git',str(engine_dir)], check=True)
else: subprocess.run(['git','fetch','--depth','1','origin','tag','v0.1.2'], cwd=engine_dir, check=True); subprocess.run(['git','checkout','--force','v0.1.2'], cwd=engine_dir, check=True)


In [ ]:
# Use the published CUDA binary for NVIDIA T4 (SM75); do not compile from source.
import hashlib
release_url='https://github.com/pwilkin/acestep.cpp/releases/download/v0.1.2/acestep-cuda-linux-x64.tar.gz'
expected_sha256='2f06f56d5fb0ab1a5eaa59f02133d5b9f83205cb4d36db11cc66fa2367c2d33d'
archive=Path('/kaggle/working/acestep-cuda-linux-x64.tar.gz')
subprocess.run(['curl','-fL','--retry','5','--retry-all-errors','--connect-timeout','30','-o',str(archive),release_url], check=True)
hasher=hashlib.sha256()\nwith archive.open('rb') as stream:\n    for chunk in iter(lambda: stream.read(8*1024*1024), b''):\n        hasher.update(chunk)\ndigest=hasher.hexdigest()
if digest != expected_sha256: raise RuntimeError(f'ACE-Step release SHA-256 mismatch: {digest}')
build_dir=engine_dir/'build'
build_dir.mkdir(parents=True,exist_ok=True)
subprocess.run(['tar','-xzf',str(archive),'-C',str(build_dir)], check=True)
binary=build_dir/'ace-server'
if not binary.is_file(): raise RuntimeError('Release archive did not contain build/ace-server')
binary.chmod(binary.stat().st_mode | 0o111)
ldd=subprocess.run(['ldd',str(binary)],text=True,capture_output=True,check=True)
print(ldd.stdout,flush=True)
if 'not found' in ldd.stdout: raise RuntimeError('ACE-Step release has unresolved shared-library dependencies')
subprocess.run([str(binary),'--help'],cwd=engine_dir,check=True,stdout=subprocess.DEVNULL)
print('ACE_STEP_BINARY_READY: v0.1.2; SHA256 verified; CUDA Linux x64',flush=True)


In [ ]:
# Q8_0 Turbo model set. ace-server loads models on first request and retains them with --keep-loaded.
subprocess.run(['bash','-lc','./models.sh --lm 4B --quant Q8_0'], cwd=engine_dir, check=True)
print('GGUF files:', [p.name for p in (engine_dir/'models').glob('*.gguf')], flush=True)

In [ ]:
repo_dir=Path('/kaggle/working/kaggle-music-generator')
if not repo_dir.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/unisocsu/kaggle-music-generator.git',str(repo_dir)], check=True)
subprocess.run([sys.executable,str(repo_dir/'kaggle'/'session_worker.py'),'--repo','unisocsu/kaggle-music-generator','--poll-seconds','15','--idle-seconds','180','--max-seconds',str(4*60*60+30*60)], cwd=engine_dir, check=True)
print('Session finished. Output files:', [p.name for p in Path('/kaggle/working/output').glob('*.mp3')], flush=True)